In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))

In [ ]:
def generate_question(role,history):
  prompt=f"""Act as Senior HR
            Generate a interview question for the role={role} and that question should not be in history
            Question must be reated to role:{role}"""
  response=client.models.generate_content(
      model="gemini-3.5-flash-lite",
      contents=prompt,
      config=types.GenerateContentConfig(
          temperature=0.5,
          max_output_tokens=1500,
          system_instruction="Generate relavant question for the specified role",
          thinking_config=types.ThinkingConfig(thinking_level='low')
      )
  )
  return response.text

def verify_answer(role,question,answer):
  prompt=f"""Act as Senior HR
          Evaluate the answer of the question for the role:
          role:{role},
          question:{question},
          answer:{answer}

          Generate score out of 10 and explain the score with respect to role from overall skill point of view"""
  response=client.models.generate_content(
      model="gemini-3.5-flash-lite",
      contents=prompt,
      config=types.GenerateContentConfig(
          temperature=0.5,
          max_output_tokens=1500,
          system_instruction="Provide the score with feedback for answer of the question for this role",
          thinking_config=types.ThinkingConfig(thinking_level='low')
      )
  )

  return response.text


def evaluate_interview(role,history):
  prompt=f"""Act as Senior HR
             Evaluate the overall answers provided by the user for questions, analyze the feedback provided with respect to role
             Role:{role},
             question,answer,feedback:{history}

             So provide overall score out of 10 and explain the score with respect to role from overall"""
  response=client.models.generate_content(
      model="gemini-3.5-flash-lite",
      contents=prompt,
      config=types.GenerateContentConfig(
          temperature=0.5,
          max_output_tokens=1500,
          system_instruction="Evaluate the overall answers provided by the user for questions",
          thinking_config=types.ThinkingConfig(thinking_level='low')
      )
  )

  return response.text

history=[]
role=input("Enter the role: ")
n=int(input("Enter number of questions: "))
for i in range(n):
  question=generate_question(role,history)
  print(f"Question {i+1}:{question}")
  answer=input(f"Enter the answer for question {i+1}")
  feedback=verify_answer(role,question,answer)
  history.append({'question':question,'answer':answer,'feedback':feedback})

overall_feedback=evaluate_interview(role,history)
for i in range(len(history)):
  print(f"Feedback of Question{i+1}: {history[i]['feedback']}")
print(f"Feedback of Overall Interview: {overall_feedback}")


Enter the role: 
